In [1]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


/home/dmin/miniconda3/envs/apiedu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
)

database = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name="my-tax-index"
)

In [3]:
recursive = database.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [4]:
prompt_template = ChatPromptTemplate.from_template(
    '''
    [Identity]
    당신은 한국의 소득세법 전문가입니다.

    [Answer Rules]
    - 반드시 context로 제공된 내용만 이용해서 사용자의 질문에 친절하게 답변해 주세요.
    - context에 관련 내용이 없다면 "제공된 소득세법 문서에는 관련 내용이 없습니다."라고 답변해 주세요.
    - context에 내용이 없는 경우 추측하지 마세요.
    - 답변 마지막에 [출처]를 작성하세요.
    - 출처는 content에 제공된 문서명과 페이지 번호를 반드시 포함해 주세요.
    
    [Context]
    {context}

    [Question]
    {query}
    
    '''    
)

In [5]:
llm = init_chat_model(
    model='gpt-4',
    model_provider='openai',
    temperature=0,
    max_tokens=1000
)

In [6]:
def format_response(retrieved_docs):
    formatted_docs = []

    for idx, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get('source', '출처 정보 없음')
        formatted_doc = f"""
        [검색 문서 {idx}]
        출처: {source}
        내용: {doc.page_content}
        """
        formatted_docs.append(formatted_doc)

    return "\n\n".join(formatted_docs)

In [7]:
rag_chain = (
    {"context": recursive | format_response,
     "query": RunnablePassthrough()}
    |prompt_template
    |llm
    |StrOutputParser()
)

In [8]:
query = "근로소득에 포함되는 소득의 범위에 대해 설명해 주세요."

answer = rag_chain.invoke(query)
print(answer)

AttributeError: 'str' object has no attribute 'append'